In [1]:
import zipfile
import glob
import polars as pl

In [2]:
def parse_line(x: str, indices: list[int], delimiter: str = "|"):
    x = x.split(delimiter)
    return [x[idx] for idx in indices]

In [3]:
zip_files = glob.glob("./data/raw/*.zip")

# Carrier, flight number, date, departure time, tail number
INDICES = [4, 5, 8, 10, 25]

all_data = []

for zf in zip_files:
    print(zf)
    with zipfile.ZipFile(zf, "r") as f:
        data_file = f.namelist()[0]
        file_data = f.read(data_file)
        for line in file_data.splitlines():
            all_data.append(parse_line(line.decode("utf-8"), INDICES))

./data/raw/ONTIME.TD.202204.REL01.14JUN2022.zip
./data/raw/ONTIME.TD.202312.REL01.31JAN2024.zip
./data/raw/ONTIME.TD.202104.REL01.27MAY2021.zip
./data/raw/ONTIME.TD.202302.REL01.17APR2023.zip
./data/raw/ONTIME.TD.202210.REL01.30NOV2022.zip
./data/raw/ONTIME.TD.202108.REL01.30SEP2021.zip
./data/raw/ONTIME.TD.202401.REL01.11MAR2024.zip
./data/raw/ONTIME.TD.202111.REL01.04JAN2022.zip
./data/raw/ONTIME.TD.202402.REL01.01APR2024.zip
./data/raw/ONTIME.TD.202308.REL01.02OCT2023.zip
./data/raw/ONTIME.TD.202311.REL01.04JAN2024.zip
./data/raw/ONTIME.TD.202406.REL01.06AUG2024.zip
./data/raw/ONTIME.TD.202109.REL01.02NOV2021.zip
./data/raw/ONTIME.TD.202408.REL01.02OCT2024.zip
./data/raw/ONTIME.TD.202105.REL02.01JUL2021.zip
./data/raw/ONTIME.TD.202209.REL01.03NOV2022.zip
./data/raw/ONTIME.TD.202405.REL02.24JUL2024.zip
./data/raw/ONTIME.TD.202208.REL01.27SEP2022.zip
./data/raw/ONTIME.TD.202310.REL01.06DEC2023.zip
./data/raw/ONTIME.TD.202301.REL02.16MAR2023.zip
./data/raw/ONTIME.TD.202411.REL01.07JAN2

In [5]:
flights_2024 = pl.DataFrame(
    all_data, 
    orient="row",
    schema={
        "carrier": pl.String,
        "flight_number": pl.String,
        "date": pl.String,
        "departure_time": pl.String,
        "tail_number": pl.String,
    }
)

In [6]:
# lol registraction
registrations = (
    pl.read_csv(
        "./data/raw/AircraftRegistractionData.csv",
        schema_overrides={"Nnumber": pl.String}
    )
    .select(
        pl.concat_str(pl.lit("N"), pl.col("Nnumber")).alias("tail_number"),
        pl.col("Model").alias("aircraft_type"),
    )
)

In [7]:
flights_2024 = flights_2024.join(registrations, on="tail_number", how="left")
flights_2024.write_parquet("./data/supplemental.parquet")

In [8]:
flights_2024.group_by("tail_number").len().sort("len", descending=True)

tail_number,len
str,u32
"""""",103814
"""N486HA""",11228
"""N493HA""",11156
"""N492HA""",11006
"""N476HA""",10920
…,…
"""N568WN""",1
"""N707SA""",1
"""N152NK""",1


In [9]:
flights_2024.group_by("aircraft_type").len().sort("len", descending=True)

aircraft_type,len
str,u32
"""ERJ 170-200 LR""",2855661
"""737-7H4""",1949062
null,1932780
"""CL-600-2D24""",1578347
"""A320-232""",1197894
…,…
"""DA 40""",2
"""MBB-BK 117 C-2""",2
"""A75""",1
